In [1]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch
from TimexNormUtils import TemporalDataset, compute_metrics, collator

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_EPOCHS = 15
LR = 5e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 16
MAX_NEW_TOK  = 64

cleandata_path = "D:\\GeoTKG\\cleandata\\normalise\\"
train = TemporalDataset(cleandata_path + "train.json")
eval = TemporalDataset(cleandata_path + "eval.json")
train_loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collator(b, tokenizer))
eval_loader = torch.utils.data.DataLoader(eval, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collator(b, tokenizer))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

steps_per_epoch = len(train_loader)
eval_steps_per_epoch = len(eval_loader)

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
tokenizer.add_special_tokens({"additional_special_tokens": ["DCT:", "TYPE:", "TEXT:", "SPAN:"]})
model.resize_token_embeddings(len(tokenizer))

Embedding(50269, 1024)

In [3]:
history = {"loss": [], "eval_loss":[], "acc":[], "relax_acc":[]}
model.to(device)
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    model.eval()
    eval_loss = 0.0
    decoded_preds, decoded_labels = [], []

    with torch.no_grad():
        for batch in eval_loader:
            # keep a copy of labels for loss and for decoding ground truth
            labels_copy = batch["labels"].clone()
            batch = {k: v.to(device) for k, v in batch.items()}

            # loss
            out = model(**batch)
            eval_loss += out.loss.item()

            # predictions
            gen_ids = model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_new_tokens=MAX_NEW_TOK,
                num_beams=4,
                early_stopping=True
            )
            decoded_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))

            # decode ground truth labels (replace -100 → pad for decoding)
            labels_copy[labels_copy == -100] = tokenizer.pad_token_id
            decoded_labels.extend(tokenizer.batch_decode(labels_copy, skip_special_tokens=True))

    avg_eval_loss = eval_loss / max(1, len(eval_loader))
    metrics = compute_metrics(decoded_labels, decoded_preds)

    history["loss"].append(total_loss)
    history["eval_loss"].append(avg_eval_loss)
    history["acc"].append(metrics["accuracy strict"])
    history["relax_acc"].append(metrics["accuracy relaxed"])
    print(f"EPOCH: {epoch+1} LOSS: {total_loss/steps_per_epoch:.4f} ACC: {metrics['accuracy strict']} RACC: {metrics['accuracy relaxed']}")
    if (epoch+1)%5==0:
        torch.save({'model_state_dict': model.state_dict()}, f"results/time_norm/time_norm_epoch{epoch+1}.pt")
        

OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 24.98 GiB is allocated by PyTorch, and 311.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)